# 03 - Entrenar Modelo 2 (multi-label, 5 patógenos)

Una hoja puede tener varias enfermedades simultáneamente. Por eso entrenamos sigmoid + binary cross-entropy (focal) en vez de softmax.

Clases: `bacterianas`, `fungicas`, `plagas_insectos`, `roya`, `virales`.

Pipeline:
1. EfficientNetB0 backbone + cabeza sigmoid(N)
2. BinaryFocalCrossentropy(γ=2, α=0.25) maneja desbalance mejor que BCE simple
3. CutMix entre clases distintas: enseña co-ocurrencia con etiquetas multi-hot
4. Label smoothing 0.05
5. Calibración de threshold por clase maximizando F1 en validación
6. Salida: model2_pathogen.keras + thresholds.json

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib

In [ ]:
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = (224, 224)
BATCH = 16
EPOCHS_P1 = 20
EPOCHS_P2 = 35
LR_P1 = 1e-3
LR_P2 = 5e-6
CUTMIX_PROB = 0.5
LABEL_SMOOTH = 0.05

DATA = Path("./splits/clasificacion_patogeno")
OUT = Path("./outputs")
OUT.mkdir(exist_ok=True)

In [ ]:
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input

TRAIN_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.3),
    A.GaussNoise(var_limit=(5, 25), p=0.25),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=0, p=0.4),
    A.CoarseDropout(max_holes=6, max_height=32, max_width=32, fill_value=0, p=0.2),
])
VAL_AUG = A.Compose([])


def multi_hot(label_idx: int, num_classes: int) -> np.ndarray:
    vector = np.zeros(num_classes, dtype=np.float32)
    vector[label_idx] = 1.0
    return vector


def smooth_labels(y: np.ndarray, epsilon: float) -> np.ndarray:
    return y * (1 - epsilon) + epsilon / y.shape[-1]


def cutmix_pair(img_a, y_a, img_b, y_b):
    lam = np.random.beta(1.0, 1.0)
    h, w = img_a.shape[:2]
    cut_w = int(w * np.sqrt(1 - lam))
    cut_h = int(h * np.sqrt(1 - lam))
    cx = np.random.randint(w)
    cy = np.random.randint(h)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, w)
    y2 = min(cy + cut_h // 2, h)
    mixed = img_a.copy()
    mixed[y1:y2, x1:x2] = img_b[y1:y2, x1:x2]
    real_lam = 1 - ((x2 - x1) * (y2 - y1) / (w * h))
    y_mixed = y_a * real_lam + y_b * (1 - real_lam)
    return mixed, y_mixed


class MultiLabelLeafSequence(Sequence):
    def __init__(self, directory, img_size, batch_size, augment, shuffle, cutmix=False):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.cutmix = cutmix
        self.class_indices = {}
        self.samples = []
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for idx, name in enumerate(classes):
            self.class_indices[name] = idx
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / name).glob(ext):
                    self.samples.append((str(fp), idx))
        self.n = len(self.samples)
        self.num_classes = len(classes)
        if shuffle:
            self._shuffle()

    def _shuffle(self):
        np.random.shuffle(self.samples)

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, batch_idx):
        batch = self.samples[batch_idx * self.batch_size:(batch_idx + 1) * self.batch_size]
        images, labels = [], []
        for fp, cls in batch:
            img = np.array(Image.open(fp).convert("RGB").resize(self.img_size))
            if self.augment:
                img = TRAIN_AUG(image=img)["image"]
            else:
                img = VAL_AUG(image=img)["image"]
            images.append(img)
            labels.append(multi_hot(cls, self.num_classes))
        images = np.stack(images).astype(np.float32)
        labels = np.stack(labels)
        if self.cutmix and self.augment and np.random.rand() < CUTMIX_PROB:
            images, labels = self._cutmix_batch(images, labels)
        labels = smooth_labels(labels, LABEL_SMOOTH)
        return preprocess_input(images), labels

    def _cutmix_batch(self, images, labels):
        perm = np.random.permutation(len(images))
        out_imgs = np.empty_like(images)
        out_labels = np.empty_like(labels)
        for i in range(len(images)):
            out_imgs[i], out_labels[i] = cutmix_pair(
                images[i], labels[i], images[perm[i]], labels[perm[i]]
            )
        return out_imgs, out_labels

    def on_epoch_end(self):
        if self.shuffle:
            self._shuffle()


train_gen = MultiLabelLeafSequence(
    DATA / "train", IMG_SIZE, BATCH, augment=True, shuffle=True, cutmix=True
)
val_gen = MultiLabelLeafSequence(
    DATA / "val", IMG_SIZE, BATCH, augment=False, shuffle=False, cutmix=False
)
NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = [k for k, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]
print(f"Train: {train_gen.n} | Val: {val_gen.n} | Clases: {CLASS_NAMES}")
with open(OUT / "class_indices_model2_pathogen.json", "w") as f:
    json.dump(train_gen.class_indices, f)

In [ ]:
def build_multilabel_model(num_classes: int) -> tf.keras.Model:
    base = tf.keras.applications.EfficientNetB0(
        weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = False
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    output = tf.keras.layers.Dense(num_classes, activation="sigmoid")(x)
    return tf.keras.Model(base.input, output, name="model2_pathogen_multilabel")


FOCAL_LOSS = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25, from_logits=False)
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name="bin_acc"),
    tf.keras.metrics.AUC(multi_label=True, name="auc"),
    tf.keras.metrics.Precision(name="prec"),
    tf.keras.metrics.Recall(name="rec"),
]

model = build_multilabel_model(NUM_CLASSES)
model.compile(optimizer=tf.keras.optimizers.Adam(LR_P1), loss=FOCAL_LOSS, metrics=METRICS)
model.summary()

In [ ]:
callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=str(OUT / "model2_pathogen_p1_best.keras"), monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

print("=== FASE 1: cabeza congelada ===")
h1 = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P1, callbacks=callbacks_p1, verbose=1)

In [ ]:
for layer in model.layers:
    layer.trainable = True
for layer in model.layers[:-30]:
    layer.trainable = False
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(LR_P2), loss=FOCAL_LOSS, metrics=METRICS)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=str(OUT / "model2_pathogen_p2_best.keras"), monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=4, min_lr=1e-8, verbose=1),
]

print("=== FASE 2: fine-tuning ultimos 30 layers ===")
h2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P2, initial_epoch=len(h1.history["loss"]),
    callbacks=callbacks_p2, verbose=1,
)
model.save(OUT / "model2_pathogen.keras")
print("Modelo guardado en", OUT / "model2_pathogen.keras")

## Calibración de thresholds por clase

Para cada clase buscamos el threshold que maximiza F1 sobre validación. El servidor de inferencia lo usa para decidir qué clases están activas en cada parche.

In [ ]:
from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score


def collect_predictions(generator, model):
    y_true, y_pred = [], []
    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        y_true.append(y_batch)
        y_pred.append(model.predict(x_batch, verbose=0))
    return np.concatenate(y_true), np.concatenate(y_pred)


y_true_val, y_pred_val = collect_predictions(val_gen, model)
y_true_binary = (y_true_val > 0.5).astype(int)


def best_threshold_for_class(y_true_col, y_pred_col):
    precisions, recalls, thresholds = precision_recall_curve(y_true_col, y_pred_col)
    f1_values = 2 * precisions * recalls / np.clip(precisions + recalls, 1e-9, None)
    best_idx = int(np.argmax(f1_values[:-1])) if len(f1_values) > 1 else 0
    return float(thresholds[best_idx]) if len(thresholds) else 0.5


thresholds = {}
report = {}
for i, name in enumerate(CLASS_NAMES):
    t = best_threshold_for_class(y_true_binary[:, i], y_pred_val[:, i])
    thresholds[name] = round(t, 3)
    pred_col = (y_pred_val[:, i] >= t).astype(int)
    report[name] = {
        "threshold": round(t, 3),
        "f1": round(float(f1_score(y_true_binary[:, i], pred_col, zero_division=0)), 3),
        "ap": round(float(average_precision_score(y_true_binary[:, i], y_pred_val[:, i])), 3),
    }
    print(f"{name:20s}  thr={t:.3f}  F1={report[name]['f1']:.3f}  AP={report[name]['ap']:.3f}")

with open(OUT / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)
with open(OUT / "calibration_report.json", "w") as f:
    json.dump(report, f, indent=2)

f1_macro = float(np.mean([v["f1"] for v in report.values()]))
map_macro = float(np.mean([v["ap"] for v in report.values()]))
print(f"\nF1 macro: {f1_macro:.3f}  |  mAP: {map_macro:.3f}")

In [ ]:
loss_h = h1.history["loss"] + h2.history["loss"]
vloss = h1.history["val_loss"] + h2.history["val_loss"]
auc = h1.history["auc"] + h2.history["auc"]
vauc = h1.history["val_auc"] + h2.history["val_auc"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(loss_h) + 1)
div = len(h1.history["loss"])
ax1.plot(ep, loss_h, "b-", label="train")
ax1.plot(ep, vloss, "r-", label="val")
ax1.axvline(div, color="gray", linestyle="--", label="fine-tune")
ax1.set_xlabel("Epoca"); ax1.set_ylabel("Focal loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, auc, "b-", label="train")
ax2.plot(ep, vauc, "r-", label="val")
ax2.axvline(div, color="gray", linestyle="--")
ax2.set_xlabel("Epoca"); ax2.set_ylabel("AUC multi-label"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "model2_pathogen_curves.png", dpi=120)
plt.show()